In [ ]:
import sys
import matplotlib.pyplot as plt
import numpy as np
import os

try:
    # パスを追加
    sys.path.append('C:/Users/daiki/sokendai/N2O/experiment data/annealing')
    
    # モジュールをインポート
    import Read_tiffimage
    import FFTVAv2
    
    # ファイルパスの設定
    file_path = 'C:/Users/daiki/sokendai/N2O/experiment data/annealing/zipdata/1550/'
    save_path = 'C:/Users/daiki/sokendai/N2O/experiment data/annealing/zipdata/1550/'
    
    # ファイルの存在確認
    if os.path.exists(file_path):
        print(f"処理開始: {file_path}")
        
        # データ読み込み
        x1, y1, z1 = Read_tiffimage.tiffr(file_path)
        print(f"データ読み込み完了: {z1.shape}")
        
        # FFT処理
        Zs_tg, Awav = FFTVAv2.FFTimage(z1)
        print(f"FFT処理完了: {Zs_tg.shape}")
        
        # プロット
        plt.figure(figsize=(10, 8))
        contour = plt.contourf(x1, y1, Zs_tg[300, :, :])
        plt.colorbar(contour)
        plt.title('FFT Result')
        plt.show()
        
        # 保存
        np.save(save_path + 'tg.npy', Zs_tg)
        np.save(save_path + 'wavelength.npy', Awav)
        print("処理完了")
        
    else:
        print(f"ディレクトリが見つかりません: {file_path}")
        
except Exception as e:
    print(f"エラーが発生しました: {e}")
    import traceback
    traceback.print_exc()


In [ ]:

# 必要なライブラリのインポート
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
import pandas as pd

# フォントサイズの設定
rcParams['font.size'] = 20
rcParams['axes.labelsize'] = 26
rcParams['axes.titlesize'] = 20
rcParams['xtick.labelsize'] = 30
rcParams['ytick.labelsize'] = 30
rcParams['legend.fontsize'] = 24
rcParams['figure.titlesize'] = 22

# ベースライン補正関数
def rolling_ball_baseline(y, window_size=150, iterations=5):
    peak_width = estimate_peak_width(y)
    adaptive_window = max(peak_width * 3, window_size)
    y_ext = np.pad(y, (adaptive_window//2, adaptive_window//2), mode='reflect')
    baseline = y_ext.copy()
    window_sizes = [adaptive_window // 2, adaptive_window, adaptive_window * 2]
    for size in window_sizes:
        for _ in range(iterations):
            rolling_min = pd.Series(baseline).rolling(window=size, center=True, min_periods=1).min()
            rolling_max = pd.Series(rolling_min).rolling(window=size, center=True, min_periods=1).max()
            baseline = rolling_max.values
    baseline = pd.Series(baseline).rolling(window=5, center=True, min_periods=1).mean().values
    return baseline[adaptive_window//2:-adaptive_window//2]

# ピーク幅推定関数
def estimate_peak_width(y):
    gradient = np.gradient(y)
    peaks = np.where(np.diff(np.sign(gradient)))[0]
    if len(peaks) >= 2:
        peak_widths = np.diff(peaks)
        return int(np.median(peak_widths))
    return 50

# データファイルパスの設定
base_path = 'C:/Users/daiki/sokendai/N2O/experiment data/UV/zipdata/'
path_ref = base_path + 'ref/'
time_point = '1014'

# 波長範囲の設定（5–15 µm → 5–10 µm に限定）
yw_nm = np.arange(5000, 15000, 10)
yw = yw_nm / 1000
mask = (yw >= 5.0) & (yw <= 10.0)
yw = yw[mask]

# 空間分解能パラメータ
xst, yst = 150, 130
sbin = 28
x0 = np.arange(xst, xst+11*sbin, sbin)
y0 = np.arange(yst, yst+11*sbin, sbin)

# 中心領域（i6-j6）のピクセル位置
i, j = 5, 5
pix_x, pix_y = x0[j], y0[i]

# データ読み込みと吸収スペクトル計算
path_sam = base_path + time_point + '/'
Zs_tg = np.load(path_sam + 'tg.npy')
Zs_ref = np.load(path_ref + 'ref.npy')
Absorb = np.log10(Zs_ref / Zs_tg)

# 中心領域の平均スペクトル抽出（波長方向にマスク適用）
spectrum_full = np.average(Absorb[:, pix_y:pix_y+sbin, pix_x:pix_x+sbin], axis=(1,2))
spectrum = spectrum_full[mask]

# ベースライン補正
baseline = rolling_ball_baseline(spectrum)
corrected_spectrum = spectrum - baseline

# グラフ描画（5–10 µm）- Raw, Baseline, Corrected の3つを表示
fig, ax = plt.subplots(figsize=(14, 10))

# Raw spectrum（生スペクトル）
ax.plot(yw, spectrum, color='gray', linewidth=2.0, label='Raw spectrum', alpha=0.8)

# Baseline（ベースライン）
ax.plot(yw, baseline, color='red', linewidth=2.0, label='Baseline', linestyle='--')

# Corrected spectrum（補正後スペクトル）
ax.plot(yw, corrected_spectrum, color='blue', linewidth=2.5, label='After Baseline spectrum', alpha=0.9)

ax.set_xlim([5, 10])
ax.set_ylim([-0.2, 1.2])
ax.grid(True, alpha=0.3)
ax.set_xticks(np.arange(5, 11, 1))
ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_xlabel('Wavelength [μm]', fontsize=28, weight='bold')
ax.set_ylabel('Absorbance', fontsize=28, weight='bold')
ax.legend(fontsize=24, loc='upper right', frameon=True,
          edgecolor='black', fancybox=True, shadow=True)
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.5, linewidth=1)
plt.tight_layout()
plt.show()

In [ ]:
# 必要なライブラリのインポート
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
import pandas as pd
from scipy.signal import find_peaks, savgol_filter

# フォントサイズの設定
rcParams['font.size'] = 14
rcParams['axes.labelsize'] = 16
rcParams['axes.titlesize'] = 18
rcParams['xtick.labelsize'] = 28
rcParams['ytick.labelsize'] = 28
rcParams['legend.fontsize'] = 12
rcParams['figure.titlesize'] = 20

# 改良版ベースライン補正関数
def refined_rolling_ball_baseline(y, base_window=100, iterations=3):
    peak_width = estimate_peak_width(y)
    adaptive_window = max(peak_width * 2, base_window)

    y_ext = np.pad(y, (adaptive_window//2, adaptive_window//2), mode='reflect')
    baseline = y_ext.copy()

    window_sizes = [adaptive_window // 2, adaptive_window, adaptive_window * 2]
    for size in window_sizes:
        for _ in range(iterations):
            rolling_min = pd.Series(baseline).rolling(window=size, center=True, min_periods=1).min()
            rolling_max = pd.Series(rolling_min).rolling(window=size, center=True, min_periods=1).max()
            baseline = rolling_max.values

    smoothed_baseline = savgol_filter(baseline, window_length=31, polyorder=3)
    return smoothed_baseline[adaptive_window//2:-adaptive_window//2]

# ピーク幅推定関数（改良版）
def estimate_peak_width(y):
    peaks, _ = find_peaks(y, prominence=0.05)
    if len(peaks) >= 2:
        widths = np.diff(peaks)
        return int(np.median(widths))
    return 50

# データファイルのパス設定
base_path = 'C:/Users/daiki/sokendai/N2O/experiment data/annealing/zipdata/'
path_ref = base_path + 'ref/'

# 温度と時間ポイントの設定
selected_time_points = ['1413', '1440', '1513']
selected_temperature = [110.2, 121.6, 104.0]
temp_labels = ['110.2 K', '121.6 K', '104.0 K']
colors = ['red', 'blue', 'green']

# 波長範囲の設定
yw_nm = np.arange(5000,15000,10)
yw = yw_nm / 1000
nw = len(yw)

# 空間分解能パラメータ
xst, yst = 150, 130
sbin = 28
x0 = np.arange(xst, xst+11*sbin, sbin)
y0 = np.arange(yst, yst+11*sbin, sbin)

# 中心領域のインデックス
center_i, center_j = 5, 5

# 単一プロットの準備
plt.figure(figsize=(10, 6))

# 各温度の補正スペクトルをプロット
for temp_idx, (temp, time_point, temp_label, color) in enumerate(zip(selected_temperature, selected_time_points, temp_labels, colors)):
    print(f"Processing temperature {temp} K...")
    
    path_sam = base_path + time_point + '/'
    solid_ref = np.load(path_ref + 'ref.npy')
    solid_tg = np.load(path_sam + 'tg.npy')
    
    solid_absorb = np.log10(solid_ref / solid_tg)
    
    pix_x, pix_y = x0[center_j], y0[center_i]
    raw_spectrum = np.average(solid_absorb[:, pix_y:pix_y+sbin, pix_x:pix_x+sbin], axis=(1,2))
    
    baseline = refined_rolling_ball_baseline(raw_spectrum)
    corrected_spectrum = raw_spectrum - baseline
    
    plt.plot(yw, corrected_spectrum, color=color, linewidth=2.5, label=f'{temp_label}')

# 軸設定とラベル
plt.xlim([5, 10])
plt.ylim([-0.2, 1.2])
plt.xticks([5, 6, 7, 8, 9, 10])
plt.yticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2])
plt.xlabel('Wavelength [μm]', fontsize=32)
plt.ylabel('Absorbance ', fontsize=32)
plt.legend(fontsize=24)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import matplotlib.patches as patches

# ===============================
# データパス設定
# ===============================
base_path = 'C:/Users/daiki/sokendai/N2O/experiment data/UV/zipdata/'
ref_path = os.path.join(base_path, 'ref/')
sample_path = os.path.join(base_path, '1014/')

# ===============================
# 対象波長（nm）
# ===============================
WAVELENGTH_V1 = 7750
WAVELENGTH_2V2 = 8600
WAVELENGTH_TORSION = 7270

# ===============================
# 基本関数
# ===============================
def load_npy(file_path):
    if os.path.exists(file_path):
        return np.load(file_path)
    else:
        print(f"ファイルが見つかりません: {file_path}")
        return None

def calculate_absorbance(ref, sample):
    epsilon = 1e-10
    return np.log10((ref + epsilon) / (sample + epsilon))

def calculate_ratio_map(numerator, denominator):
    epsilon = 1e-10
    return (numerator + epsilon) / (denominator + epsilon)

def calculate_average_absorbance(data_map):
    overall_avg = np.mean(data_map)

    circle_radius = 154
    x_center, y_center = 304, 264
    ny, nx = data_map.shape
    x, y = np.meshgrid(np.arange(nx), np.arange(ny))
    distance = np.sqrt((x - x_center) ** 2 + (y - y_center) ** 2)
    mask = distance <= circle_radius

    circle_avg = np.mean(data_map[mask])
    return overall_avg, circle_avg

# ===============================
# グリッド描画
# ===============================
def add_grid(ax):
    x_start, y_start = 150, 130
    x_bins, y_bins, bin_size = 11, 11, 28

    for i in range(x_bins):
        for j in range(y_bins):
            x = x_start + bin_size * i
            y = y_start + bin_size * j

            if i == 5 and j == 5:
                edge_color = 'yellow'
                line_style = 'solid'
                line_width = 4
            else:
                edge_color = 'red'
                line_style = 'dotted'
                line_width = 2

            rect = patches.Rectangle(
                (x, y), bin_size, bin_size,
                lw=line_width, ec=edge_color,
                linestyle=line_style, fill=False
            )
            ax.add_patch(rect)

            if j == y_bins - 1:
                ax.text(
                    x + bin_size / 2, y + bin_size + 5, f"j{i + 1}",
                    ha='center', va='bottom',
                    fontsize=16, color='orange', fontweight='bold'
                )
            if i == 0:
                ax.text(
                    x_start - 15, y + bin_size / 2, f"i{y_bins - j}",
                    ha='right', va='center',
                    fontsize=16, color='orange', fontweight='bold'
                )

# ===============================
# 横並びマップ描画（カラーバー完全分離）
# ===============================
def plot_maps_row(data_maps, titles, mode="absorbance", figsize=(26, 8)):
    n = len(data_maps)
    ny, nx = data_maps[0].shape

    fig, axes = plt.subplots(1, n, figsize=figsize, sharey=True)

    if n == 1:
        axes = [axes]

    for ax, data_map, title in zip(axes, data_maps, titles):
        if mode == "ratio":
            vmin, vmax = 0.0, 1.0
        else:
            vmin, vmax = 0.0, float(np.max(data_map))

        im = ax.imshow(
            data_map,
            cmap='coolwarm',
            origin='lower',
            extent=[0, nx, 0, ny],
            aspect='auto',
            vmin=vmin,
            vmax=vmax
        )

        ax.set_title(title, fontsize=30)
        ax.set_xlabel('X', fontsize=26)
        ax.tick_params(labelsize=24)

        add_grid(ax)

    axes[0].set_ylabel('Y', fontsize=26)

    # ===== カラーバーを「もっと右」に完全分離 =====
    cax = fig.add_axes([0.94, 0.15, 0.02, 0.7])  
    # [left, bottom, width, height]

    cbar = fig.colorbar(im, cax=cax)
    cbar.ax.tick_params(labelsize=24)

    if mode == "ratio":
        cbar.set_label('Ratio', fontsize=28)
        cbar.set_ticks([0.0, 0.5, 1.0])
    else:
        cbar.set_label('Absorbance', fontsize=28)

    plt.show()

# ===============================
# メイン処理
# ===============================
def main():
    print("=== UV吸光度 & 比マップ解析 ===")

    ref = load_npy(os.path.join(ref_path, 'ref.npy'))
    sample = load_npy(os.path.join(sample_path, 'tg.npy'))

    if ref is None or sample is None:
        return
    if ref.shape != sample.shape:
        print("参照データとサンプルデータの形状が一致しません。")
        return

    wavelength_array = np.arange(5000, 15000, 10)
    absorbance_cube = calculate_absorbance(ref, sample)

    idx_v1 = np.where(wavelength_array == WAVELENGTH_V1)[0]
    idx_2v2 = np.where(wavelength_array == WAVELENGTH_2V2)[0]
    idx_torsion = np.where(wavelength_array == WAVELENGTH_TORSION)[0]

    if len(idx_v1) == 0 or len(idx_2v2) == 0 or len(idx_torsion) == 0:
        print("指定波長が見つかりません。")
        return

    map_v1 = absorbance_cube[idx_v1[0]]
    map_2v2 = absorbance_cube[idx_2v2[0]]
    map_torsion = absorbance_cube[idx_torsion[0]]

    # ===== 吸光度マップ =====
    plot_maps_row(
        [map_v1, map_2v2, map_torsion],
        ["N₂O ice v₁", "N₂O ice 2v₂", "N₂O ice torsion"],
        mode="absorbance",
        figsize=(30, 9)
    )

    torsion_v1 = calculate_ratio_map(map_torsion, map_v1)
    v2_v1 = calculate_ratio_map(map_2v2, map_v1)

    print("\n=== 平均値 ===")
    o, c = calculate_average_absorbance(torsion_v1)
    print(f"Torsion / v₁  全体平均: {o:.4f}  円内平均: {c:.4f}")

    o, c = calculate_average_absorbance(v2_v1)
    print(f"2v₂ / v₁      全体平均: {o:.4f}  円内平均: {c:.4f}")

    # ===== 比マップ =====
    plot_maps_row(
        [torsion_v1, v2_v1],
        ["N₂O ice torsion / v₁", "N₂O ice 2v₂ / v₁"],
        mode="ratio",
        figsize=(22, 9)
    )

# ===============================
# 実行
# ===============================
if __name__ == "__main__":
    main()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import matplotlib.patches as patches

# ===============================
# データ設定
# ===============================
base_path = 'C:/Users/daiki/sokendai/N2O/experiment data/annealing/zipdata/'
ref_path = os.path.join(base_path, 'ref/')

selected_points = ['1413', '1440', '1513']
selected_temperatures = [110.2, 121.6, 104.0]  # 列

target_wavelengths = {
    'v₁': 7750,
    '2v₂': 8600,
    'torsion': 7270
}

# ===============================
# 関数群
# ===============================
def load_npy(file_path):
    if os.path.exists(file_path):
        return np.load(file_path)
    else:
        print(f"ファイルが見つかりません: {file_path}")
        return None

def calculate_absorbance(ref, sample):
    epsilon = 1e-10
    return np.log10((ref + epsilon) / (sample + epsilon))

def add_grid(ax):
    x_start, y_start = 150, 130
    x_bins, y_bins, bin_size = 11, 11, 28

    for i in range(x_bins):
        for j in range(y_bins):
            x = x_start + bin_size * i
            y = y_start + bin_size * j

            if i == 5 and j == 5:
                edge_color = 'yellow'
                lw = 4
                ls = 'solid'
            else:
                edge_color = 'red'
                lw = 2
                ls = 'dotted'

            rect = patches.Rectangle(
                (x, y),
                bin_size,
                bin_size,
                lw=lw,
                ec=edge_color,
                linestyle=ls,
                fill=False
            )
            ax.add_patch(rect)

# ===============================
# メイン処理
# ===============================
def main():
    print("=== N₂O アニーリング吸光度解析（3×3 マップ） ===")

    ref_data = load_npy(os.path.join(ref_path, 'ref.npy'))
    if ref_data is None:
        return

    wavelength_array = np.arange(5000, 15000, 10)

    # 吸光度データをまとめて保持
    absorbance_data = {}

    for idx, point in enumerate(selected_points):
        sample = load_npy(os.path.join(base_path, point, 'tg.npy'))
        if sample is None or sample.shape != ref_data.shape:
            print(f"{point} をスキップ")
            continue

        cube = calculate_absorbance(ref_data, sample)

        for label, target_nm in target_wavelengths.items():
            wl_idx = np.where(wavelength_array == target_nm)[0]
            if len(wl_idx) == 0:
                continue

            absorbance_data[(label, idx)] = cube[wl_idx[0]]

    # ===============================
    # 描画
    # ===============================
    fig, axes = plt.subplots(
        3, 3,
        figsize=(24, 22),
        constrained_layout=True
    )

    vmax = max(np.max(v) for v in absorbance_data.values())

    for row, (label, _) in enumerate(target_wavelengths.items()):
        for col, temp in enumerate(selected_temperatures):
            ax = axes[row, col]
            amap = absorbance_data[(label, col)]

            im = ax.imshow(
                amap,
                cmap='coolwarm',
                origin='lower',
                aspect='auto',
                vmin=0,
                vmax=vmax
            )

            add_grid(ax)

            # 行ラベル（左）
            if col == 0:
                ax.set_ylabel(label, fontsize=40, labelpad=20)

            # 列ラベル（上）
            if row == 0:
                ax.set_title(f"{temp} K", fontsize=42, pad=20)

            ax.set_xticks([])
            ax.set_yticks([])

    # カラーバー（右に1本）
    cbar = fig.colorbar(im, ax=axes, shrink=0.85, pad=0.02)
    cbar.set_label("Absorbance", fontsize=38)
    cbar.ax.tick_params(labelsize=34)

    plt.show()

if __name__ == "__main__":
    main()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import math
import matplotlib.patches as patches

# =========================
# データパス設定
# =========================
base_path = 'C:/Users/daiki/sokendai/N2O/experiment data/UV/zipdata/'
ref_path = os.path.join(base_path, 'ref/')
sample_path = os.path.join(base_path, '1014/')

# =========================
# 基本関数
# =========================
def load_npy(file_path):
    if os.path.exists(file_path):
        return np.load(file_path)
    else:
        print(f"ファイルが見つかりません: {file_path}")
        return None


def calculate_absorbance(ref, sample):
    epsilon = 1e-10
    return np.log10((ref + epsilon) / (sample + epsilon))


def calculate_thickness_map(absorbance_map):
    AV_16 = 1.2e-17     # cm^2 / molecule
    Z = 44              # g / mol
    NA = 6.02e23        # molecules / mol
    rho = 1.263         # g / cm^3
    dv = 18             # / cm

    d_map_um = (absorbance_map * dv * math.log(10) * Z) / (NA * AV_16 * rho) * 1e4
    d_map_um[d_map_um < 0] = 0

    N_map_mol_cm2 = (absorbance_map * dv) / AV_16

    return d_map_um, N_map_mol_cm2

# =========================
# グリッド描画（円なし）
# =========================
def add_grid():
    x_start, y_start = 150, 130
    x_bins, y_bins, bin_size = 11, 11, 28

    ax = plt.gca()

    for i in range(x_bins):
        for j in range(y_bins):
            x = x_start + bin_size * i
            y = y_start + bin_size * j

            if i == 5 and j == 5:
                edge_color = 'yellow'
                line_style = 'solid'
                line_width = 4
            else:
                edge_color = 'red'
                line_style = 'dotted'
                line_width = 2

            rect = patches.Rectangle(
                (x, y), bin_size, bin_size,
                lw=line_width, ec=edge_color,
                linestyle=line_style, fill=False
            )
            ax.add_patch(rect)

            if j == y_bins - 1:
                ax.text(
                    x + bin_size / 2, y + bin_size + 5,
                    f"j{i+1}", ha='center', va='bottom',
                    fontsize=16, color='orange', fontweight='bold'
                )

            if i == 0:
                ax.text(
                    x_start - 15, y + bin_size / 2,
                    f"i{y_bins - j}", ha='right', va='center',
                    fontsize=16, color='orange', fontweight='bold'
                )

# =========================
# プロット関数
# =========================
def plot_thickness_map(thickness_map, title="Thickness Map"):
    ny, nx = thickness_map.shape
    plt.figure(figsize=(12, 10))

    im = plt.imshow(
        thickness_map, cmap='coolwarm',
        origin='lower', extent=[0, nx, 0, ny],
        aspect='auto', vmin=0, vmax=np.max(thickness_map)
    )

    plt.xlabel('X', fontsize=30)
    plt.ylabel('Y', fontsize=30)
    plt.title(title, fontsize=34)
    plt.xticks(fontsize=30)
    plt.yticks(fontsize=30)

    add_grid()

    cbar = plt.colorbar(im, shrink=0.8, pad=0.03)
    cbar.ax.tick_params(labelsize=28)
    cbar.set_label('Thickness (µm)', fontsize=30)
    cbar.ax.set_yticklabels([f"{t:.2f}" for t in cbar.get_ticks()])

    plt.tight_layout()
    plt.show()


def plot_n_map(n_map, title="Column density map"):
    ny, nx = n_map.shape
    plt.figure(figsize=(12, 10))

    im = plt.imshow(
        n_map, cmap='coolwarm',
        origin='lower', extent=[0, nx, 0, ny],
        aspect='auto', vmin=0, vmax=np.max(n_map)
    )

    plt.xlabel('X', fontsize=30)
    plt.ylabel('Y', fontsize=30)
    plt.title(title, fontsize=34)
    plt.xticks(fontsize=30)
    plt.yticks(fontsize=30)

    add_grid()

    cbar = plt.colorbar(im, shrink=0.8, pad=0.03)
    cbar.ax.tick_params(labelsize=28)
    cbar.set_label('Column density (molecules/cm$^2$)', fontsize=30)
    cbar.ax.set_yticklabels([f"{t:.2e}" for t in cbar.get_ticks()])

    plt.tight_layout()
    plt.show()

# =========================
# main
# =========================
def main():
    print("=== N₂O氷厚さマップ計算（7.75 µm） ===")

    ref_file = os.path.join(ref_path, 'ref.npy')
    sample_file = os.path.join(sample_path, 'tg.npy')

    ref_data = load_npy(ref_file)
    sample_data = load_npy(sample_file)

    if ref_data is None or sample_data is None:
        return

    if ref_data.shape != sample_data.shape:
        print("参照データとサンプルデータの形状が一致しません。")
        return

    wavelength_array = np.arange(5000, 15000, 10)

    absorbance_cube = calculate_absorbance(ref_data, sample_data)

    target_wavenumber = 7750
    index = np.where(wavelength_array == target_wavenumber)[0]

    if len(index) == 0:
        print("指定波数が見つかりません。")
        return

    A7750 = absorbance_cube[index[0]]

    d_map_um, N_map_mol_cm2 = calculate_thickness_map(A7750)

    print("\n厚さマップ統計:")
    print(f"  最小値: {np.min(d_map_um):.3f} µm")
    print(f"  最大値: {np.max(d_map_um):.3f} µm")
    print(f"  平均値: {np.mean(d_map_um):.3f} µm")
    print(f"  標準偏差: {np.std(d_map_um):.3f} µm")

    print("\nN_map_(molecule/cm^2) 統計:")
    print(f"  最小値: {np.min(N_map_mol_cm2):.3e}")
    print(f"  最大値: {np.max(N_map_mol_cm2):.3e}")
    print(f"  平均値: {np.mean(N_map_mol_cm2):.3e}")
    print(f"  標準偏差: {np.std(N_map_mol_cm2):.3e}")

    plot_thickness_map(d_map_um)
    plot_n_map(N_map_mol_cm2)

# =========================
# デバッグ用：単一ピクセル確認
# =========================
absorbance_map = np.random.rand(512, 512)

x, y = 304, 264
dv = 18

A = absorbance_map[y, x]
log10_term = A * math.log(10)
full_term = A * dv * math.log(10)

print("\n=== 単一ピクセル計算チェック ===")
print(f"座標 ({x}, {y}) の吸光度 A = {A:.5f}")
print(f"A * log(10) = {log10_term:.5f}")
print(f"A * dv * log(10) = {full_term:.5f}")

# =========================
# 実行
# =========================
if __name__ == "__main__":
    main()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# =========================
# データパス設定
# =========================
ref_path = 'C:/Users/daiki/sokendai/N2O/experiment data/UV/zipdata/ref/'
time_data = {
    '60min':  'C:/Users/daiki/sokendai/N2O/experiment data/UV/zipdata/1114/',
    '70min':  'C:/Users/daiki/sokendai/N2O/experiment data/UV/zipdata/1124/',
    '80min':  'C:/Users/daiki/sokendai/N2O/experiment data/UV/zipdata/1134/',
    '90min':  'C:/Users/daiki/sokendai/N2O/experiment data/UV/zipdata/1144/',
    '120min': 'C:/Users/daiki/sokendai/N2O/experiment data/UV/zipdata/1214/',
    '510min': 'C:/Users/daiki/sokendai/N2O/experiment data/UV/zipdata/1814/'
}

# 解析対象
wavelength_data = {
    7750: 'N₂O v₁',
    5340: 'NO'
}

time_order = ['60min', '70min', '80min', '90min', '120min', '510min']

# =========================
# 基本関数
# =========================
def load_npy(file_path):
    if os.path.exists(file_path):
        return np.load(file_path)
    else:
        print(f"⚠ ファイルが見つかりません: {file_path}")
        return None

def calculate_absorbance(ref, sample):
    epsilon = 1e-10
    return np.log10((ref + epsilon) / (sample + epsilon))

# =========================
# グリッド描画
# =========================
def add_grid_only(ax):
    x_start, y_start = 150, 130
    x_bins, y_bins, bin_size = 11, 11, 28

    for i in range(x_bins):
        for j in range(y_bins):
            x = x_start + bin_size * i
            y = y_start + bin_size * j

            if i == 5 and j == 5:
                edge_color = 'yellow'
                line_style = 'solid'
                line_width = 4
            else:
                edge_color = 'red'
                line_style = 'dotted'
                line_width = 2

            rect = patches.Rectangle(
                (x, y),
                bin_size,
                bin_size,
                lw=line_width,
                ec=edge_color,
                linestyle=line_style,
                fill=False
            )
            ax.add_patch(rect)

# =========================
# 2×3 プロット本体
# =========================
def plot_2x3_absorbance(wavelength, molecule_label):
    print(f"\n=== {molecule_label} ({wavelength} nm) 解析開始 ===")

    wavelength_array = np.arange(5000, 15000, 10)
    index = np.where(wavelength_array == wavelength)[0]
    if len(index) == 0:
        print("⚠ 指定波長が見つかりません")
        return
    index = index[0]

    ref_data = load_npy(os.path.join(ref_path, 'ref.npy'))
    if ref_data is None:
        return

    absorbance_maps = []
    vmax = 0

    for t in time_order:
        sample_data = load_npy(os.path.join(time_data[t], 'tg.npy'))
        if sample_data is None or sample_data.shape != ref_data.shape:
            absorbance_maps.append(None)
            continue

        absorbance = calculate_absorbance(ref_data, sample_data)[index]
        absorbance_maps.append(absorbance)
        vmax = max(vmax, np.max(absorbance))

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()

    for ax, t, amap in zip(axes, time_order, absorbance_maps):
        if amap is None:
            ax.set_title(f"{t}\n(no data)", fontsize=20)
            ax.axis('off')
            continue

        im = ax.imshow(
            amap,
            cmap='coolwarm',
            origin='lower',
            aspect='auto',
            vmin=0,
            vmax=vmax
        )
        ax.set_title(t, fontsize=22)
        ax.set_xticks([])
        ax.set_yticks([])
        add_grid_only(ax)

    # ===== カラーバーを「かなり右」に配置 =====
    cbar_ax = fig.add_axes([0.93, 0.15, 0.02, 0.7])
    cbar = fig.colorbar(im, cax=cbar_ax)

    if molecule_label == 'N₂O v₁':
        cbar.set_label('N₂O v₁ Absorbance', fontsize=24)
    else:
        cbar.set_label('NO Absorbance', fontsize=24)

    cbar.ax.tick_params(labelsize=18)

    # suptitle は完全に削除
    plt.show()

# =========================
# main
# =========================
def main():
    for wavelength, label in wavelength_data.items():
        plot_2x3_absorbance(wavelength, label)

if __name__ == "__main__":
    main()



In [ ]:
# 必要なライブラリのインポート
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
import pandas as pd
import matplotlib.cm as cm

# フォントサイズの設定
rcParams['font.size'] = 16
rcParams['axes.labelsize'] = 18
rcParams['axes.titlesize'] = 20
rcParams['xtick.labelsize'] = 16
rcParams['ytick.labelsize'] = 16
rcParams['legend.fontsize'] = 14
rcParams['figure.titlesize'] = 22

# ベースライン補正関数
def rolling_ball_baseline(y, window_size=150, iterations=5):
    peak_width = estimate_peak_width(y)
    adaptive_window = max(peak_width * 3, window_size)
    y_ext = np.pad(y, (adaptive_window//2, adaptive_window//2), mode='reflect')
    baseline = y_ext.copy()
    window_sizes = [adaptive_window // 2, adaptive_window, adaptive_window * 2]
    for size in window_sizes:
        for _ in range(iterations):
            rolling_min = pd.Series(baseline).rolling(window=size, center=True, min_periods=1).min()
            rolling_max = pd.Series(rolling_min).rolling(window=size, center=True, min_periods=1).max()
            baseline = rolling_max.values
    baseline = pd.Series(baseline).rolling(window=5, center=True, min_periods=1).mean().values
    return baseline[adaptive_window//2:-adaptive_window//2]

# ピーク幅推定関数
def estimate_peak_width(y):
    gradient = np.gradient(y)
    peaks = np.where(np.diff(np.sign(gradient)))[0]
    if len(peaks) >= 2:
        peak_widths = np.diff(peaks)
        return int(np.median(peak_widths))
    return 50

# データファイルパスの設定
base_path = 'C:/Users/daiki/sokendai/N2O/experiment data/UV/zipdata/'
path_ref = base_path + 'ref/'

# 測定時間点の設定
selected_times = [0, 30, 60, 70, 80, 90, 120, 510]
selected_time_points = ['1014', '1044', '1114', '1124', '1134', '1144', '1214', '1837']

# 波長範囲の設定
yw_nm = np.arange(5000, 15000, 10)
yw = yw_nm / 1000
nw = len(yw)

# 空間分解能パラメータ
xst, yst = 150, 130
sbin = 28
x0 = np.arange(xst, xst+11*sbin, sbin)
y0 = np.arange(yst, yst+11*sbin, sbin)

# 中心領域（i6-j6）のピクセル位置
i, j = 5, 5
pix_x, pix_y = x0[j], y0[i]

# カラーマップ設定（虹色）
cmap = cm.get_cmap('rainbow')
rainbow_colors = [cmap(i/7) for i in range(8)]

# スペクトル保存用辞書
time_spectra = {}
diff_spectra = {}

# スペクトル処理ループ
print("Processing i6-j6 transmission spectra...")
for time_idx, (time, time_point) in enumerate(zip(selected_times, selected_time_points)):
    print(f"Processing spectrum for time {time} min...")
    path_sam = base_path + time_point + '/'
    Zs_tg = np.load(path_sam + 'tg.npy')
    Zs_ref = np.load(path_ref + 'ref.npy')
    Absorb = np.log10(Zs_ref / Zs_tg)
    spectrum = np.average(Absorb[:, pix_y:pix_y+sbin, pix_x:pix_x+sbin], axis=(1,2))
    baseline = rolling_ball_baseline(spectrum)
    corrected_spectrum = spectrum - baseline
    time_spectra[time] = corrected_spectrum

# 差スペクトルの計算（0分を基準）
corrected_spectrum_0 = time_spectra[0]
for time in selected_times:
    diff_spectra[time] = time_spectra[time] - corrected_spectrum_0

# 1. 補正スペクトルの描画
fig1, ax1 = plt.subplots(figsize=(14, 10))
for time_idx, time in enumerate(selected_times):
    ax1.plot(yw, time_spectra[time], color=rainbow_colors[time_idx],
             linewidth=2.5, label=f'{time} min', alpha=0.85)
ax1.set_xlim([5, 15])
ax1.set_ylim([-0.1, 1.3])
ax1.grid(True, alpha=0.3)
ax1.set_xticks(np.arange(5, 16, 1))
ax1.set_yticks([0, 0.5, 1.0])
ax1.set_xlabel('Wavelength [μm]', fontsize=28, weight='bold')
ax1.set_ylabel('Absorbance', fontsize=28, weight='bold')
ax1.legend(fontsize=16, loc='upper right', ncol=2, frameon=True,
           edgecolor='black', fancybox=True, shadow=True)
ax1.axhline(y=0, color='gray', linestyle='-', alpha=0.5, linewidth=1)
plt.tight_layout()
plt.show()

# 2. 差スペクトルの描画
fig2, ax2 = plt.subplots(figsize=(14, 10))
for time_idx, time in enumerate(selected_times):
    ax2.plot(yw, diff_spectra[time], color=rainbow_colors[time_idx],
             linewidth=2.5, label=f'{time} min', alpha=0.85)
ax2.set_xlim([5, 15])
ax2.set_ylim([-0.3, 0.3])
ax2.grid(True, alpha=0.3)
ax2.set_xticks(np.arange(5, 16, 1))
ax2.set_yticks(np.arange(-0.3, 0.31, 0.1))
ax2.set_xlabel('Wavelength [μm]', fontsize=28, weight='bold')
ax2.set_ylabel('ΔAbsorbance', fontsize=28, weight='bold')
ax2.legend(fontsize=16, loc='upper right', ncol=2, frameon=True,
           edgecolor='black', fancybox=True, shadow=True)
ax2.axhline(y=0, color='gray', linestyle='-', alpha=0.5, linewidth=1)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
import pandas as pd
from scipy.optimize import curve_fit
from scipy.integrate import solve_ivp

# フォント設定
rcParams.update({
    'font.size': 16,
    'axes.labelsize': 18,
    'axes.titlesize': 20,
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'legend.fontsize': 16,
    'figure.titlesize': 22
})

# ベースライン補正
def rolling_ball_baseline(y, window_size=150, iterations=5):
    peak_width = estimate_peak_width(y)
    adaptive_window = max(peak_width * 3, window_size)
    y_ext = np.pad(y, (adaptive_window//2, adaptive_window//2), mode='reflect')
    baseline = y_ext.copy()
    for size in [adaptive_window // 2, adaptive_window, adaptive_window * 2]:
        for _ in range(iterations):
            rolling_min = pd.Series(baseline).rolling(window=size, center=True, min_periods=1).min()
            rolling_max = pd.Series(rolling_min).rolling(window=size, center=True, min_periods=1).max()
            baseline = rolling_max.values
    baseline = pd.Series(baseline).rolling(window=5, center=True, min_periods=1).mean().values
    return baseline[adaptive_window//2:-adaptive_window//2]

# ピーク幅推定
def estimate_peak_width(y):
    gradient = np.gradient(y)
    peaks = np.where(np.diff(np.sign(gradient)))[0]
    return int(np.median(np.diff(peaks))) if len(peaks) >= 2 else 50

# 吸光度抽出
def extract_absorbance_at_wavelength(spectrum, wavelength_array, target_wavelength):
    idx = np.argmin(np.abs(wavelength_array - target_wavelength))
    return spectrum[idx]

# 反応速度論的微分方程式システム
def reaction_kinetics(t, y, k1, k2):
    """
    反応方程式:
    d[N2O]/dt = -k1[N2O] + k2[N2][O]
    d[N2]/dt = k1[N2O] - k2[N2][O]
    d[O]/dt = k1[N2O] - k2[N2][O]
    """
    N2O, N2, O = y
    
    dN2O_dt = -k1 * N2O + k2 * N2 * O
    dN2_dt = k1 * N2O - k2 * N2 * O
    dO_dt = k1 * N2O - k2 * N2 * O
    
    return [dN2O_dt, dN2_dt, dO_dt]

# フィッティング用の関数（N2Oの濃度のみを返す）
def n2o_kinetic_model(t, k1, k2, N2O_0, N2_0, O_0):
    """
    微分方程式を解いてN2O濃度の時間変化を返す
    """
    # 初期条件
    y0 = [N2O_0, N2_0, O_0]
    
    # 微分方程式を解く
    sol = solve_ivp(
        lambda t_val, y_val: reaction_kinetics(t_val, y_val, k1, k2),
        [0, t[-1]],
        y0,
        t_eval=t,
        method='RK45',
        rtol=1e-8,
        atol=1e-10
    )
    
    # N2O濃度のみを返す
    return sol.y[0]

# 波長データ（nm → µm）
wavelength_data = {
    7750: 'N₂O v₁',
    5340: 'NO',
    6310: 'N₂O₃',
}
target_wavelengths = {name: wl / 1000 for wl, name in wavelength_data.items()}

# 時間点とファイル名
selected_times = [0, 30, 60, 70, 80, 90, 120, 180, 240, 300, 360, 420, 480, 510]
selected_time_points = ['1014', '1044', '1114', '1124', '1134', '1144', '1214',
                        '1314', '1414', '1514', '1614', '1714', '1814', '1837']

# 波長軸
yw_nm = np.arange(5000, 15000, 10)
yw = yw_nm / 1000

# 空間分解能
xst, yst, sbin = 150, 130, 28
x0 = np.arange(xst, xst+11*sbin, sbin)
y0 = np.arange(yst, yst+11*sbin, sbin)
i, j = 5, 5
pix_x, pix_y = x0[j], y0[i]

# データパス
base_path = 'C:/Users/daiki/sokendai/N2O/experiment data/UV/zipdata/'
path_ref = base_path + 'ref/'

# 吸光度格納
absorbance_data = {name: [] for name in target_wavelengths}

# 各時間点で処理
for time, time_point in zip(selected_times, selected_time_points):
    path_sam = base_path + time_point + '/'
    Zs_tg = np.load(path_sam + 'tg.npy')
    Zs_ref = np.load(path_ref + 'ref.npy')
    Absorb = np.log10(Zs_ref / Zs_tg)
    spectrum = np.average(Absorb[:, pix_y:pix_y+sbin, pix_x:pix_x+sbin], axis=(1,2))
    baseline = rolling_ball_baseline(spectrum)
    corrected_spectrum = spectrum - baseline
    for name, wl in target_wavelengths.items():
        absorbance = extract_absorbance_at_wavelength(corrected_spectrum, yw, wl)
        absorbance_data[name].append(absorbance)

# N₂O v₁ 反応速度論的フィッティング
fit_name = 'N₂O v₁'
fit_start_index = selected_times.index(60)
selected_times_fit = np.array(selected_times[fit_start_index:])
selected_times_fit_relative = selected_times_fit - 60  # 60分を0とする
y_fit = np.array(absorbance_data[fit_name][fit_start_index:])

# 初期パラメータの推定
# 吸光度を相対濃度として扱う（正規化）
N2O_0_init = y_fit[0]  # 初期N2O濃度
N2_0_init = 0.1 * N2O_0_init  # 初期N2濃度（推定）
O_0_init = 0.1 * N2O_0_init   # 初期O濃度（推定）
k1_init = 0.01  # 分解速度定数の初期値 (min^-1)
k2_init = 0.001  # 再結合速度定数の初期値 (min^-1)

# パラメータの境界設定
bounds = (
    [1e-6, 1e-6, 0.1*N2O_0_init, 0, 0],  # 下限
    [1.0, 1.0, 2*N2O_0_init, N2O_0_init, N2O_0_init]  # 上限
)

try:
    # 反応速度論的フィッティング
    popt_kinetic, pcov_kinetic = curve_fit(
        n2o_kinetic_model,
        selected_times_fit_relative,
        y_fit,
        p0=[k1_init, k2_init, N2O_0_init, N2_0_init, O_0_init],
        bounds=bounds,
        maxfev=10000
    )
    
    k1_fit, k2_fit, N2O_0_fit, N2_0_fit, O_0_fit = popt_kinetic
    
    # フィッティング曲線の生成
    time_fit_fine = np.linspace(0, 450, 200)
    n2o_fit = n2o_kinetic_model(time_fit_fine, *popt_kinetic)
    time_fit_abs = time_fit_fine + 60
    
    # 色設定
    colors = ['crimson', 'blue', 'green', 'orange', 'purple', 'brown', 'red', 'teal', 'darkcyan', 'magenta', 'gold', 'gray', 'navy', 'olive']
    color_map = dict(zip(target_wavelengths.keys(), colors))
    
    # グラフ描画（点のみ＋N₂O v₁反応速度論的フィット）
    fig, ax = plt.subplots(figsize=(14, 9))
    for name in target_wavelengths:
        ax.plot(selected_times, absorbance_data[name], 'o', color=color_map[name], markersize=8,
                markerfacecolor='white', markeredgewidth=2,
                label=f'{name} ({target_wavelengths[name]:.2f} µm)')
    
    # N₂O v₁ 反応速度論的フィッティング曲線
    ax.plot(time_fit_abs, n2o_fit, '--', color=color_map[fit_name], linewidth=3, 
            label=f'{fit_name} Kinetic Fit')

    #    # グラフ装飾
    ax.grid(True, color='gray', alpha=0.3)
    ax.set_xlim([-10, 520])
    ax.set_xticks(np.arange(0, 541, 60))
    ax.set_xticklabels([str(t) for t in np.arange(0, 541, 60)], fontsize=24)
    ax.tick_params(axis='y', labelsize=24)
    ax.set_xlabel('Time (min)', fontsize=28, weight='bold')
    ax.set_ylabel('Absorbance', fontsize=28, weight='bold')
    ax.legend(loc='upper right', fontsize=16)
    plt.tight_layout()
    plt.show()
    
    # 数値出力（反応速度論的フィッティング結果）
    y_pred = n2o_kinetic_model(selected_times_fit_relative, *popt_kinetic)
    r2_kinetic = 1 - np.sum((y_fit - y_pred)**2) / np.sum((y_fit - np.mean(y_fit))**2)
    
    # 半減期の計算（近似的）
    half_value = N2O_0_fit / 2
    time_fine = np.linspace(0, 1000, 10000)
    n2o_fine = n2o_kinetic_model(time_fine, *popt_kinetic)
    half_life_idx = np.argmin(np.abs(n2o_fine - half_value))
    half_life_min = time_fine[half_life_idx]
    half_life_sec = half_life_min * 60
    
    print(f"\n=== {fit_name} 反応速度論的フィッティング結果 ===")
    print(f"k1 (分解速度定数): {k1_fit/60:.6f} /s")
    print(f"k2 (再結合速度定数): {k2_fit/60:.6f} /s")
    print(f"初期N2O濃度: {N2O_0_fit:.4f}")
    print(f"初期N2濃度: {N2_0_fit:.4f}")
    print(f"初期O濃度: {O_0_fit:.4f}")
    print(f"Half-life: {half_life_min:.1f} min ({half_life_sec:.0f} s)")
    print(f"R²: {r2_kinetic:.4f}")
    print(f"Initial (60 min): {y_fit[0]:.4f}, Final (510 min): {y_fit[-1]:.4f}")
    print(f"Decrease ratio: {(1 - y_fit[-1]/y_fit[0])*100:.1f}%")
    
    # パラメータの標準誤差
    param_errors = np.sqrt(np.diag(pcov_kinetic))
    print(f"\n=== パラメータの標準誤差 ===")
    print(f"k1 error: ±{param_errors[0]/60:.6f} /s")
    print(f"k2 error: ±{param_errors[1]/60:.6f} /s")
    print(f"N2O_0 error: ±{param_errors[2]:.4f}")
    print(f"N2_0 error: ±{param_errors[3]:.4f}")
    print(f"O_0 error: ±{param_errors[4]:.4f}")

except Exception as e:
    print(f"反応速度論的フィッティングでエラーが発生しました: {e}")
    print("単純な指数減衰フィッティングにフォールバックします...")
    
    # 指数減衰関数（フォールバック）
    def exponential_decay(t, A, k, C):
        return A * np.exp(-k * t) + C
    
    A_init = y_fit[0] - y_fit[-1]
    k_init = 0.01
    C_init = y_fit[-1]
    popt, _ = curve_fit(exponential_decay, selected_times_fit_relative, y_fit,
                        p0=[A_init, k_init, C_init], maxfev=5000)
    A_fit, k_fit, C_fit = popt
    time_fit_rel = np.linspace(0, 450, 100)
    time_fit_abs = time_fit_rel + 60
    absorbance_fit = exponential_decay(time_fit_rel, A_fit, k_fit, C_fit)
    
    # グラフ描画（点のみ＋N₂O v₁フィット）
    fig, ax = plt.subplots(figsize=(14, 9))
    for name in target_wavelengths:
        ax.plot(selected_times, absorbance_data[name], 'o', color=color_map[name], markersize=8,
                markerfacecolor='white', markeredgewidth=2,
                label=f'{name} ({target_wavelengths[name]:.2f} µm)')

    # N₂O v₁ フィッティング曲線
    ax.plot(time_fit_abs, absorbance_fit, '--', color=color_map[fit_name], linewidth=2, label=f'{fit_name} Fit')

    # グラフ装飾
    ax.grid(True, color='gray', alpha=0.3)
    ax.set_xlim([-10, 520])
    ax.set_xticks(np.arange(0, 541, 60))
    ax.set_xticklabels([str(t) for t in np.arange(0, 541, 60)], fontsize=24)
    ax.tick_params(axis='y', labelsize=24)
    ax.set_xlabel('Time (min)', fontsize=28, weight='bold')
    ax.set_ylabel('Absorbance', fontsize=28, weight='bold')
    ax.legend(loc='upper right', fontsize=16)
    plt.tight_layout()
    plt.show()
    
    # 数値出力（指数減衰フィッティング結果）
    r2 = 1 - np.sum((y_fit - exponential_decay(selected_times_fit_relative, *popt))**2) / np.sum((y_fit - np.mean(y_fit))**2)
    half_life_min = np.log(2) / k_fit
    half_life_sec = np.log(2) / (k_fit / 60)
    print(f"\n=== {fit_name} 指数減衰フィッティング結果（フォールバック） ===")
    print(f"A: {A_fit:.4f}, k: {k_fit:.6f} /s, C: {C_fit:.4f}")
    print(f"Half-life: {half_life_min:.1f} min ({half_life_sec:.0f} s)")
    print(f"R²: {r2:.4f}")
    print(f"Initial (60 min): {y_fit[0]:.4f}, Final (510 min): {y_fit[-1]:.4f}")
    print(f"Decrease ratio: {(1 - y_fit[-1]/y_fit[0])*100:.1f}%")

# 反応速度論的解析の比較（成功した場合のみ）
try:
    if 'popt_kinetic' in locals():
        print(f"\n=== 反応速度論的モデル vs 指数減衰モデル比較 ===")
        
        # 指数減衰モデルでの再フィッティング（比較用）
        def exponential_decay_comp(t, A, k, C):
            return A * np.exp(-k * t) + C
        
        popt_exp, _ = curve_fit(exponential_decay_comp, selected_times_fit_relative, y_fit,
                               p0=[y_fit[0] - y_fit[-1], 0.01, y_fit[-1]], maxfev=5000)
        
        y_pred_exp = exponential_decay_comp(selected_times_fit_relative, *popt_exp)
        r2_exp = 1 - np.sum((y_fit - y_pred_exp)**2) / np.sum((y_fit - np.mean(y_fit))**2)
        
        print(f"反応速度論的モデル R²: {r2_kinetic:.4f}")
        print(f"指数減衰モデル R²: {r2_exp:.4f}")
        
        if r2_kinetic > r2_exp:
            print("反応速度論的モデルの方が実験データによく適合しています。")
        else:
            print("指数減衰モデルの方が実験データによく適合しています。")
            
        # AIC（赤池情報量規準）による比較
        n = len(y_fit)
        
        # 反応速度論的モデル（5パラメータ）
        rss_kinetic = np.sum((y_fit - y_pred)**2)
        aic_kinetic = n * np.log(rss_kinetic/n) + 2 * 5
        
        # 指数減衰モデル（3パラメータ）
        rss_exp = np.sum((y_fit - y_pred_exp)**2)
        aic_exp = n * np.log(rss_exp/n) + 2 * 3
        
        print(f"\nAIC比較:")
        print(f"反応速度論的モデル AIC: {aic_kinetic:.2f}")
        print(f"指数減衰モデル AIC: {aic_exp:.2f}")
        
        if aic_kinetic < aic_exp:
            print("AIC基準では反応速度論的モデルが優れています。")
        else:
            print("AIC基準では指数減衰モデルが優れています。")
            
except:
    print("モデル比較でエラーが発生しました。")

print("\n=== フィッティング解析完了 ===")